# Oral Biofilm × COMETS: SIIRI インターン デモ

**目的**: COMETS (dFBA) × TMCMC (Bayesian UQ) を使い、インプラント周囲バイオフィルムの
健常 ↔ dysbiotic 転換を定量的にモデル化する。

**NIFE / SIIRI との接点**:
- Szafranski ら (Dieckow 2024, *npj Biofilms Microbiomes*): 12患者 × 3時点の口腔インプラントメタゲノム
- Joshi ら (2025/2026): peri-implantitis バイオマーカー (16S + metatranscriptomics)
- このノートはその **in silico counterpart** を構築する

---
### パイプライン
```
Dieckow 2024 (ENA: PRJEB71108)            Joshi 2025 (SRA: PRJNA1192962)
  ↓  species abundance (16S)                ↓  metatranscriptomics
  Initial conditions (φ_i)              Metabolic activity (flux proxy)
         ↓                                         ↓
   COMETS dFBA ←── AGORA GEM (5 species) ──────────┘
         ↓
   Biomass trajectory + metabolite fluxes
         ↓
   Hamilton ODE fitting (TMCMC)
         ↓
   DI (Dysbiosis Index) → SIIRI infection indicator
```

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

# COMETS home detection
from pathlib import Path
for p in [os.environ.get('COMETS_HOME'), '/opt/comets_linux', str(Path.home() / 'comets_linux')]:
    if p and (Path(p) / 'bin').exists():
        os.environ['COMETS_HOME'] = p
        COMETS_AVAILABLE = True
        print(f'COMETS found: {p}')
        break
else:
    COMETS_AVAILABLE = False
    print('COMETS not found → using mock data')

# Add nife package to path
sys.path.insert(0, str(Path.cwd().parent.parent.parent))  # IKM_Hiwi root

from nife.comets.oral_biofilm import OralBiofilmComets, SPECIES, INIT_FRACTIONS
from nife.comets.tmcmc_bridge import TMCMCCoMETSBridge, SPECIES_ORDER

print('Setup complete.')

## 1. 5 菌種の生態学的役割

In [ ]:
rows = []
for key, sp in SPECIES.items():
    rows.append({
        'ID': key,
        'Species': sp['name'],
        'Role': sp['role'],
        'Secretes': ', '.join(sp['secretes'][:2]),
        'Requires': ', '.join(sp['requires'][:2]),
        'Commensal': '✓' if sp['commensal'] else '✗',
        'O2-sensitive': '✓' if sp['o2_sensitive'] else '✗',
    })
df_sp = pd.DataFrame(rows).set_index('ID')
df_sp

## 2. COMETS シミュレーション: 健常 vs peri-implantitis

In [ ]:
model = OralBiofilmComets(grid=(1, 1))

MAX_CYCLES = 500

# run() は COMETS があれば自動使用、なければ mock にフォールバック
res_h = model.run(condition='healthy',  max_cycles=MAX_CYCLES, fallback_mock=True)
res_d = model.run(condition='diseased', max_cycles=MAX_CYCLES, fallback_mock=True)
bm_h = res_h.total_biomass
bm_d = res_d.total_biomass

print(f'Mode: {"COMETS" if COMETS_AVAILABLE else "mock (COMETS not found)"}')
print('Healthy  biomass shape:', bm_h.shape)
print('Diseased biomass shape:', bm_d.shape)
bm_h.head(3)

In [ ]:
COLORS = {'So': '#2196F3', 'An': '#4CAF50', 'Vp': '#FF9800', 'Fn': '#9C27B0', 'Pg': '#F44336'}

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, bm, title in zip(axes, [bm_h, bm_d], ['Healthy (commensal)', 'Peri-implantitis (diseased)']):
    sp_cols = [c for c in bm.columns if c in SPECIES]
    totals = bm[sp_cols].sum(axis=1)
    for sp in sp_cols:
        ax.plot(bm['cycle'], bm[sp] / totals, label=sp, color=COLORS.get(sp, 'gray'), lw=2)
    ax.set_xlabel('Cycle')
    ax.set_ylabel('Relative abundance')
    ax.set_title(title)
    ax.legend()
    ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig('../oral_biofilm_abundance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: oral_biofilm_abundance.png')

## 3. Dysbiosis Index (DI) の計算

Nishioka et al. 2026 の DI 定義:
$$\text{DI}(t) = \frac{H(\phi(t))}{\log N}, \quad H = -\sum_i \phi_i \log \phi_i$$

健常バイオフィルム: DI 低 (So/An/Vp 支配 → 低エントロピー)
dysbiotic バイオフィルム: DI 高 (多種混在・Pg 増加 → 高エントロピー)

In [ ]:
di_h = model.compute_di(bm_h)
di_d = model.compute_di(bm_d)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(di_h['cycle'], di_h['DI'], label='Healthy',  color='#2196F3', lw=2)
ax.plot(di_d['cycle'], di_d['DI'], label='Diseased', color='#F44336', lw=2)
ax.axhline(0.25, color='gray', ls='--', alpha=0.5, label='DI threshold (0.25)')
ax.fill_between(di_d['cycle'], 0.25, di_d['DI'],
                where=di_d['DI'] > 0.25, alpha=0.15, color='#F44336')
ax.set_xlabel('Cycle')
ax.set_ylabel('DI (Dysbiosis Index)')
ax.set_title('DI evolution: Healthy vs Peri-implantitis')
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('../oral_biofilm_di.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Final DI — Healthy: {di_h['DI'].iloc[-1]:.3f}  |  Diseased: {di_d['DI'].iloc[-1]:.3f}")

## 4. TMCMC 後験分布 → COMETS アンサンブル

Hamilton A-matrix の TMCMC 後験サンプルを COMETS exchange reaction bounds に変換し、
DI の不確実性バンド (5-95% CI) を計算する。

In [ ]:
# --- Synthetic TMCMC posterior (proxy for real Nishioka 2026 posterior) ---
rng = np.random.default_rng(0)
N_SAMPLES = 50

# 20 parameters: 5x5 A-matrix flattened
# Healthy-like posterior: a_Pg_Pg ~ N(-0.5, 0.3)
A_mean_healthy = np.array([
    # So    An    Vp    Fn    Pg
    [-0.8, -0.2, -0.1, -0.3, -0.2],  # So
    [-0.2, -0.7, -0.1, -0.2, -0.1],  # An
    [ 0.5,  0.4, -0.6, -0.1, -0.1],  # Vp  (benefits from So/An lactate)
    [ 0.2,  0.1,  0.3, -0.5, -0.2],  # Fn
    [-0.4, -0.3, -0.2,  0.1, -0.9],  # Pg  (suppressed in healthy)
])

posterior_samples = rng.normal(
    loc=A_mean_healthy.flatten(),
    scale=0.15,
    size=(N_SAMPLES, 25)
)

param_index = {f'a_{SPECIES_ORDER[i]}_{SPECIES_ORDER[j]}': i * 5 + j
               for i in range(5) for j in range(5)}

bridge = TMCMCCoMETSBridge(model, posterior_samples, param_index)

print(f'Running {N_SAMPLES}-sample COMETS ensemble (mock)...')
results = bridge.run_ensemble(condition='healthy', n_samples=N_SAMPLES,
                               max_cycles=MAX_CYCLES, use_mock=True)
bands = TMCMCCoMETSBridge.compute_di_bands(results)
print('Done.')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

cycles = bands['cycles']
ax.fill_between(cycles, bands['di_low'], bands['di_high'],
                alpha=0.3, color='#2196F3', label='90% CI (TMCMC posterior)')
ax.plot(cycles, bands['di_median'], color='#2196F3', lw=2, label='Median DI')
ax.plot(di_h['cycle'], di_h['DI'], 'k--', lw=1.5, label='MAP estimate')
ax.axhline(0.25, color='gray', ls=':', alpha=0.7)
ax.set_xlabel('Cycle')
ax.set_ylabel('DI')
ax.set_title('DI Uncertainty (TMCMC posterior → COMETS ensemble, healthy condition)')
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('../oral_biofilm_di_uq.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Dieckow 2024 データとの接続

ENA アクセッション **PRJEB71108** (30サンプル, 12患者 × 3時点)
から処理済み abundance table があれば → 患者ごとの COMETS 初期条件として使用可能。

In [ ]:
# Simulated patient-level initial fractions (structure matches Dieckow 2024)
# Real data: FASTQ → DADA2 → HOMD → species abundance table
np.random.seed(42)

patients = [f'Patient_{c}' for c in 'ABCDEFGHIJKL']   # 12 patients
timepoints = ['Week1', 'Week2', 'Week3']

# Mock abundance data (replace with real DADA2 output)
records = []
for pt in patients:
    pg_trend = np.random.uniform(0.02, 0.30)  # patient-specific Pg level
    for t_idx, tp in enumerate(timepoints):
        pg = np.clip(pg_trend * (1 + 0.3 * t_idx), 0, 0.5)
        so = np.clip(0.40 - 0.1 * pg, 0.1, 0.6)
        an = 0.15
        vp = 0.15
        fn = 1 - so - an - vp - pg
        records.append({'patient': pt, 'timepoint': tp,
                         'So': so, 'An': an, 'Vp': vp, 'Fn': fn, 'Pg': pg})

df_patients = pd.DataFrame(records)

# Run COMETS for each patient (mock)
di_patient = {}
for _, row in df_patients[df_patients['timepoint'] == 'Week1'].iterrows():
    import nife.comets.oral_biofilm as _ob
    _ob.INIT_FRACTIONS['patient'] = {sp: row[sp] for sp in SPECIES_ORDER}
    _ob.INIT_FRACTIONS['patient'] = {sp: max(row[sp], 1e-4) for sp in SPECIES_ORDER}
    norm = sum(_ob.INIT_FRACTIONS['patient'].values())
    _ob.INIT_FRACTIONS['patient'] = {k: v/norm for k, v in _ob.INIT_FRACTIONS['patient'].items()}
    bm, _ = model.run_mock(condition='patient', max_cycles=300, noise=0.02)
    di = model.compute_di(bm)
    di_patient[row['patient']] = float(di['DI'].iloc[-1])
    _ob.INIT_FRACTIONS.pop('patient')

# Plot patient DI distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: patient DI vs initial Pg fraction
pg_vals = df_patients[df_patients['timepoint'] == 'Week1']['Pg'].values
di_vals = [di_patient[pt] for pt in patients]
axes[0].scatter(pg_vals, di_vals, c=di_vals, cmap='RdYlBu_r', s=80, zorder=3)
axes[0].set_xlabel('Initial Pg fraction (Week 1)')
axes[0].set_ylabel('Final DI')
axes[0].set_title('Patient-level DI vs Pg abundance\n(Dieckow 2024 structure)')

# Right: DI histogram
axes[1].hist(di_vals, bins=8, color='#9C27B0', alpha=0.7, edgecolor='white')
axes[1].axvline(0.25, color='red', ls='--', label='DI threshold')
axes[1].set_xlabel('Final DI')
axes[1].set_ylabel('# patients')
axes[1].set_title('DI distribution across 12 patients')
axes[1].legend()

plt.tight_layout()
plt.savefig('../oral_biofilm_patient_di.png', dpi=150, bbox_inches='tight')
plt.show()

n_high_di = sum(v > 0.25 for v in di_vals)
print(f'Patients with DI > 0.25 (dysbiosis risk): {n_high_di} / {len(patients)}')

## 6. SIIRI 感染検知への応用

DI をリアルタイム SIIRI センサー指標として定式化:

| DI 閾値 | 状態 | SIIRI アクション |
|---------|------|----------------|
| DI < 0.20 | 健常 | 経過観察 |
| 0.20 ≤ DI < 0.40 | 初期 dysbiosis | 警告・洗口指導 |
| DI ≥ 0.40 | Peri-implantitis リスク | 緊急介入 |

COMETS は DI の **力学的根拠** を提供:
- Pg の増殖は GCF 中の hemin 濃度・嫌気度に支配される
- TMCMC で患者個別の A-matrix を推定 → 個人化予測
- 次回診察前に「DI が 0.40 を超えるか」を Bayesian 予測

In [ ]:
# SIIRI infection risk indicator
fig, ax = plt.subplots(figsize=(9, 3))

ax.plot(di_h['cycle'], di_h['DI'],  color='#2196F3', lw=2, label='Healthy')
ax.plot(di_d['cycle'], di_d['DI'], color='#F44336', lw=2, label='Peri-implantitis')

ax.axhspan(0.00, 0.20, alpha=0.08, color='green',  label='Safe zone')
ax.axhspan(0.20, 0.40, alpha=0.08, color='orange', label='Early dysbiosis')
ax.axhspan(0.40, 1.00, alpha=0.08, color='red',    label='Intervention zone')

ax.set_xlabel('Cycle (time)')
ax.set_ylabel('DI')
ax.set_title('SIIRI Infection Risk Indicator based on DI')
ax.legend(loc='center right', fontsize=8)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('../oral_biofilm_siiri_indicator.png', dpi=150, bbox_inches='tight')
plt.show()

## まとめ

| コンポーネント | 本ノートでの実装 |
|---|---|
| COMETS dFBA | `OralBiofilmComets.run()` / `run_mock()` |
| Dysbiosis Index | `compute_di()` (Shannon entropy, normalized) |
| TMCMC → COMETS | `TMCMCCoMETSBridge` (exchange bounds mapping) |
| UQ バンド | `compute_di_bands()` (5-95% credible interval) |
| Dieckow 2024 data | PRJEB71108 → 患者個別初期条件 |
| SIIRI 指標 | DI 閾値ベースの 3段階アラート |

**次のステップ (インターン)**:
1. AGORA GEM をダウンロードして COMETS を正確に実行
2. PRJEB71108 FASTQ を DADA2 で処理 → 実際の abundance table で初期条件を設定
3. PRJNA1192962 (Joshi 2025) のメタトランスクリプトームでフラックス比較
4. TMCMC で患者ごとの A-matrix を推定 → 個人化 DI 予測